### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [48]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

### Data Ingestion And Processing


In [49]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [50]:
## text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])


page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [4]:
print("hello")

hello


In [51]:

print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [6]:
### load the embedding models
os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")

In [52]:
# Initialize OpenAI embeddings with the latest model

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7906.25it/s]


[-0.029035627841949463,
 0.0075597334653139114,
 0.04076479375362396,
 0.030535489320755005,
 0.051757071167230606,
 -0.01734757050871849,
 -0.030941864475607872,
 -0.06556288152933121,
 -0.033060841262340546,
 -0.009561254642903805,
 -0.09131387621164322,
 0.0473836287856102,
 0.02773960307240486,
 -0.06366036087274551,
 -0.06504747271537781,
 0.04200268164277077,
 -0.04021904617547989,
 0.028005139902234077,
 -0.02804989367723465,
 -0.053762856870889664,
 -0.005428291857242584,
 0.0060847667045891285,
 -0.07086585462093353,
 0.02593011036515236,
 0.010079517960548401,
 0.026293469592928886,
 0.038574639707803726,
 0.022752461954951286,
 -0.01670045778155327,
 0.009418061934411526,
 0.016376057639718056,
 -0.059243202209472656,
 -0.016352426260709763,
 0.04579910635948181,
 -0.061201777309179306,
 0.06117100641131401,
 -0.013366864062845707,
 -0.0003110599354840815,
 0.039353374391794205,
 -0.04183918237686157,
 -0.039276838302612305,
 -0.10107345879077911,
 -0.006085090804845095,
 -0

In [53]:
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.036539312452077866, -0.01516443770378828, 0.016432486474514008, 0.010568801313638687, 0.006010583136230707, -0.01847325637936592, 0.08546527475118637, 0.020968450233340263, 0.027815325185656548, 0.012431677430868149, -0.02937653474509716, -0.03113526478409767, 0.03491254523396492, -0.018150771036744118, -0.0649847686290741, 0.05168251320719719, -0.019606154412031174, -0.015734117478132248, -0.13371670246124268, -0.09645988047122955, -0.025471774861216545, -0.0014895533677190542, -0.0063493396155536175, -0.025820650160312653, -0.027371777221560478, 0.12268996238708496, -0.007792443502694368, -0.03852280229330063, 0.014383511617779732, -0.09218426793813705, 0.008695780299603939, 0.002613392658531666, 0.0910346582531929, -0.030313579365611076, -0.0960463434457779, 0.022289114072918892, -0.09024310857057571, -0.032947391271591187, 0.0715833529829979, -0.008893145248293877, -0.025708980858325958, -0.07913962006568909, 0.014530396088957787, -0.07420427352190018, 0.08045010268688202, 0.07

In [9]:
print(batch_embeddings[1])

[-0.02439185418188572, 0.0032444430980831385, 0.05426765978336334, -0.00667261378839612, 0.003935660235583782, -0.00795734766870737, 0.025025229901075363, -0.03203270956873894, -0.05451071634888649, -0.044702090322971344, -0.013759469613432884, 0.016061266884207726, 0.04036472737789154, -0.02026093564927578, -0.06097463145852089, 0.02065555565059185, 0.010556341148912907, -0.016264814883470535, -0.10490722209215164, -0.11068310588598251, -0.02154473401606083, -0.013036100193858147, -0.08688350021839142, 0.027151912450790405, 0.02614406682550907, 0.03964661434292793, 0.06494349986314774, 0.06547259539365768, 0.017963333055377007, -0.10655654966831207, 0.009878184646368027, -0.034961998462677, 0.030403533950448036, 0.014532767236232758, -0.11560282856225967, 0.012346191331744194, -0.06430953741073608, 0.04394598677754402, 0.01903313770890236, 0.030984872952103615, -0.015413844957947731, -0.08163447678089142, 0.012414338998496532, 0.0124236810952425, 0.06950367242097855, 0.077825434505939

In [54]:
### Compare Embedding using cosine similarity

def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [55]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.791


In [56]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.257


In [57]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.373


### Create FAISS Vector Store

In [58]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [15]:
vectorstore

In [16]:
## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [59]:
## load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [60]:
## Similarity Search 
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='9781f020-0417-426c-9c81-3d58cc10f7f3', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='9864cd25-acd9-46bd-866c-f15bd488178d', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='e677f679-ee88-48d9-98a1-982593c93671', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [61]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [62]:
### Similarity Search with score
results_with_scores=vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.343
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 1.090
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 1.154
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [20]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [21]:
### Search with metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='9b2ccb1a-a0e5-458f-92f6-9b7cc70959e2', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [22]:
len(filtered_results)

1

### Build RAG Chain With LCEL 

In [63]:
load_dotenv()

True

In [64]:
## LLM GROQ LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="groq:groq/compound-mini")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'Compound Mini', 'release_date': '2025-09-04', 'last_updated': '2025-09-04', 'open_weights': False, 'max_input_tokens': 131072, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E0F25C16E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E0F25C3230>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [65]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 437, 'total_tokens': 472, 'completion_time': 0.087823, 'completion_tokens_details': None, 'prompt_time': 0.046729, 'prompt_tokens_details': None, 'queue_time': 0.333617, 'total_time': 0.134552}, 'model_name': 'groq/compound-mini', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8f9f-8610-74d2-be1a-ac87d3e348d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 437, 'output_tokens': 35, 'total_tokens': 472})

In [30]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [31]:
## Basic retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [32]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E0EF1F7CB0>, search_kwargs={'k': 3})

In [33]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [ ]:
simple_rag_chain=(
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough() 
    }
    | simple_prompt
    | llm
    | StrOutputParser()

)

In [66]:
simple_rag_chain.invoke("What is AI?")

'Artificial Intelligence (AI) is the simulation of human intelligence in machines—systems designed to think like humans and mimic their actions.'

In [36]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [37]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [38]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [39]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [40]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [43]:
test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: AI (Artificial Intelligence) is the broad field that aims to simulate human intelligence in machines, enabling them to think, reason, and act like humans. It encompasses any technique that gives a system intelligent behavior, ranging from rule‑based systems to advanced reasoning engines, and is divided into narrow AI (focused on specific tasks) and general AI (human‑level intelligence across domains).

Machine Learning (ML) is a **subset of AI**. It specifically refers to methods that allow systems to learn patterns directly from data rather than being explicitly programmed. ML algorithms discover relationships in data to make predictions or decisions, and they include approaches such as supervised, unsupervised, and reinforcement learning.

**Key difference:**  
- **Scope:** AI is the overall discipline of creating intelligent behavior; ML is one particular way to achieve that behavior by le

In [44]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: Artificial Intelligence (AI) is the broad field that aims to simulate human intelligence in machines, encompassing any system that can think like humans and mimic their actions.  

Machine Learning (ML) is a **subset of AI** that specifically enables systems to learn from data instead of being explicitly programmed, using algorithms that find patterns in the data.  

So, while AI includes all approaches to creating intelligent behavior, ML focuses only on data‑driven learning methods within AI.

2. Streaming RAG:
Answer: Artificial Intelligence (AI) is the broad field that aims to simulate human intelligence in machines—designing systems that can think, reason, and act like humans.  
Machine Learning (ML) is a **subset of AI**; it refers specifically to techniques that enable a system to learn from data rather than being explicitly programmed, discovering patterns and making predictions. I

In [45]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1: What is machine learning?
A1: Machine learning (ML) is a branch of artificial intelligence that lets computers improve their performance on a task by learning from data rather than following explicit programming. ML algorithms automatically discover patterns and relationships in the data, enabling the system to make predictions or decisions based on new inputs.


In [46]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [47]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: Traditional programming and machine learning take opposite approaches to solving problems:

| Aspect | Traditional Programming | Machine Learning |
|--------|------------------------|------------------|
| **How the solution is created** | A developer writes explicit rules and logic that tell the computer exactly what to do for every possible input. | The system is given data and an algorithm; it automatically discovers patterns and builds a model that can make predictions or decisions. |
| **Dependence on human knowledge** | Requires the programmer to encode all domain knowledge manually. | Relies on the data to convey the knowledge; the programmer mainly designs the learning algorithm and prepares the data. |
| **Adaptability** | Changing behavior means rewriting or extending the code. | The model can be retrained with new data to adapt to changing conditions without rewriting the core algorithm. |
| **Handling complexity** | 